<a href="https://colab.research.google.com/github/Dubnitskyi/Machine_learning_labs/blob/main/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers chromadb langchain-core langchain-community

In [ ]:
pip install langchain-experimental

In [ ]:
pip install langchain-huggingface

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = """
У 2045 році світ занурився у цифрову антиутопію. Головний герой, колишній інженер Марк, живе у підземному
місті Новий Едем. Його основна мета - знайти втрачений архів даних свого батька, який містить коди доступу до системи життєзабезпечення поверхні. Під
час подорожі Марк зустрічає Сару, лідера повстанців "Тіні майбутнього". Вони виявляють, що корпорація 'Aethelgard' використовує ШІ для маніпуляції
спогадами громадян. Вирішальна битва відбувається в центрі управління корпорації, де Марк повинен вибрати між особистою помстою та порятунком
залишків екосистеми Землі. Ключовий код доступу виявився зашифрованим у ДНК Марка, що стало несподіваним відкриттям для всіх учасників
"""
fixed_no_ov_docs = RecursiveCharacterTextSplitter(chunk_size=150,chunk_overlap=0).create_documents([long_text])


In [ ]:
fixed_with_ov_docs = RecursiveCharacterTextSplitter(chunk_size=150,chunk_overlap=50).create_documents([long_text])

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

semantic_splitter = SemanticChunker(HuggingFaceEmbeddings())
chunks_semantic = semantic_splitter.split_text(long_text)

# Task 1-2

In [ ]:
questions_and_answers = {
    "У якому році світ занурився у цифрову антиутопію?": "У 2045 році",
    "Як звати головного героя і ким він був?": "Марк, колишній інженер",
    "Де живе Марк?": "у підземному місті Новий Едем",
    "Яка основна мета Марка?": "знайти втрачений архів даних свого батька, який містить коди доступу до системи життєзабезпечення поверхні",
    "Кого зустрічає Марк під час подорожі?": "Сару, лідера повстанців \"Тіні майбутнього\"",
    "Яку корпорацію виявляють Марк і Сара?": "корпорація 'Aethelgard'",
    "Що корпорація 'Aethelgard' використовує для маніпуляції спогадами громадян?": "ШІ",
    "Де відбувається вирішальна битва?": "в центрі управління корпорації",
    "Між чим повинен вибрати Марк під час вирішальної битви?": "між особистою помстою та порятунком залишків екосистеми Землі",
    "Де виявився зашифрованим ключовий код доступу?": "у ДНК Марка"
}
print("Dictionary 'questions_and_answers' created successfully.")

## Initialize ChromaDB with fixed_with_ov_docs

In [ ]:
from langchain_community.vectorstores import Chroma

embedding_function = HuggingFaceEmbeddings()
vectorstore = Chroma.from_documents(documents=fixed_with_ov_docs, embedding=embedding_function)
print("ChromaDB vector store 'vectorstore' created successfully with fixed_with_ov_docs.")

In [ ]:
retriever = vectorstore.as_retriever()

query = "Хто головний герой і де він живе?"
docs = retriever.invoke(query)

print("Retrieved documents for query:", query)
for doc in docs:
    print(doc.page_content)

In [ ]:
retrieved_contexts = []

for question, answer in questions_and_answers.items():
    retrieved_documents = retriever.invoke(question)
    retrieved_contexts.append({
        'question': question,
        'answer': answer,
        'contexts': retrieved_documents
    })

print(f"Retrieved contexts for {len(retrieved_contexts)} questions.")

In [ ]:
answer_presence = []

for item in retrieved_contexts:
    answer = item['answer'].lower()
    contexts_text = " ".join([doc.page_content.lower() for doc in item['contexts']])

    if answer in contexts_text:
        answer_presence.append(True)
    else:
        answer_presence.append(False)

print("Answer presence in retrieved contexts:")
print(answer_presence)

In [ ]:
num_answers_found = sum(answer_presence)
total_questions = len(answer_presence)

context_recall = num_answers_found / total_questions

print(f"Context Recall for fixed_with_ov_docs: {context_recall:.2f}")

## Initialize ChromaDB with fixed_no_ov_docs

In [ ]:
vectorstore_no_ov = Chroma.from_documents(documents=fixed_no_ov_docs, embedding=embedding_function)
print("ChromaDB vector store 'vectorstore_no_ov' created successfully with fixed_no_ov_docs.")

In [ ]:
retriever_no_ov = vectorstore_no_ov.as_retriever()
print("Retriever 'retriever_no_ov' created successfully.")

In [ ]:
retrieved_contexts_no_ov = []

for question, answer in questions_and_answers.items():
    retrieved_documents = retriever_no_ov.invoke(question)
    retrieved_contexts_no_ov.append({
        'question': question,
        'answer': answer,
        'contexts': retrieved_documents
    })

print(f"Retrieved contexts for {len(retrieved_contexts_no_ov)} questions using non-overlapping chunks.")

In [ ]:
answer_presence_no_ov = []

for item in retrieved_contexts_no_ov:
    answer = item['answer'].lower()
    contexts_text = " ".join([doc.page_content.lower() for doc in item['contexts']])

    if answer in contexts_text:
        answer_presence_no_ov.append(True)
    else:
        answer_presence_no_ov.append(False)

print("Answer presence in retrieved contexts (no overlap):")
print(answer_presence_no_ov)

In [ ]:
num_answers_found_no_ov = sum(answer_presence_no_ov)
total_questions_no_ov = len(answer_presence_no_ov)

context_recall_no_ov = num_answers_found_no_ov / total_questions_no_ov

print(f"Context Recall for fixed_no_ov_docs: {context_recall_no_ov:.2f}")

## Initialize ChromaDB with chunks_semantic


In [ ]:
from langchain_core.documents import Document

semantic_splitter = SemanticChunker(
    HuggingFaceEmbeddings(),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95 )
chunks_semantic = semantic_splitter.split_text(long_text)

semantic_docs = [Document(page_content=chunk) for chunk in chunks_semantic]

vectorstore_semantic = Chroma.from_documents(documents=semantic_docs, embedding=embedding_function)
print("ChromaDB vector store 'vectorstore_semantic' created successfully with semantic_docs.")

In [ ]:
retriever_semantic = vectorstore_semantic.as_retriever()
print("Retriever 'retriever_semantic' created successfully.")

In [ ]:
retrieved_contexts_semantic = []

for question, answer in questions_and_answers.items():
    retrieved_documents = retriever_semantic.invoke(question)
    retrieved_contexts_semantic.append({
        'question': question,
        'answer': answer,
        'contexts': retrieved_documents
    })

print(f"Retrieved contexts for {len(retrieved_contexts_semantic)} questions using semantic chunks.")

In [ ]:
answer_presence_semantic = []

for item in retrieved_contexts_semantic:
    answer = item['answer'].lower()
    contexts_text = " ".join([doc.page_content.lower() for doc in item['contexts']])

    if answer in contexts_text:
        answer_presence_semantic.append(True)
    else:
        answer_presence_semantic.append(False)

print("Answer presence in retrieved contexts (semantic chunks):")
print(answer_presence_semantic)

In [ ]:
num_answers_found_semantic = sum(answer_presence_semantic)
total_questions_semantic = len(answer_presence_semantic)

context_recall_semantic = num_answers_found_semantic / total_questions_semantic

print(f"Context Recall for chunks_semantic: {context_recall_semantic:.2f}")

# Task 3

## Chunk Sizes with Overlap

In [ ]:
chunk_sizes1 = []
context_recall_scores_chunk_size1 = []

for chunk_size1 in range(150, 501, 50):
    # 3a. Create RecursiveCharacterTextSplitter documents
    current_chunk_size_docs = RecursiveCharacterTextSplitter(chunk_size=chunk_size1, chunk_overlap=150).create_documents([long_text])

    # 3b. Initialize a new ChromaDB vector store
    current_chunk_size_vectorstore = Chroma.from_documents(documents=current_chunk_size_docs, embedding=embedding_function)

    # 3c. Create a retriever from this newly created vector store
    current_retriever = current_chunk_size_vectorstore.as_retriever()

    # 3d. Initialize an empty list called retrieved_contexts_current_chunk_size
    retrieved_contexts_current_chunk_size = []

    # 3e. Iterate through each question and answer
    for question, answer in questions_and_answers.items():
        # 3f. Use the current retriever to invoke the question and get the retrieved_documents
        retrieved_documents = current_retriever.invoke(question)

        # 3g. Append a dictionary containing the question, answer, and retrieved_documents
        retrieved_contexts_current_chunk_size.append({
            'question': question,
            'answer': answer,
            'contexts': retrieved_documents
        })

    # 3h. Initialize an empty list called answer_presence_current_chunk_size
    answer_presence_current_chunk_size = []

    # 3i. Iterate through each item in retrieved_contexts_current_chunk_size
    for item in retrieved_contexts_current_chunk_size:
        # 3j. Convert the answer and the page_content of all contexts to lowercase
        answer_lower = item['answer'].lower()
        contexts_text_lower = " ".join([doc.page_content.lower() for doc in item['contexts']])

        # 3k. Check if the lowercase answer is present in the concatenated lowercase contexts_text
        # 3l. Append True or False to answer_presence_current_chunk_size based on the check
        if answer_lower in contexts_text_lower:
            answer_presence_current_chunk_size.append(True)
        else:
            answer_presence_current_chunk_size.append(False)

    # 3m. Calculate the num_answers_found_current_chunk_size
    num_answers_found_current_chunk_size = sum(answer_presence_current_chunk_size)

    # 3n. Calculate the context_recall_current_chunk_size
    context_recall_current_chunk_size = num_answers_found_current_chunk_size / len(questions_and_answers)

    # 3o. Append the current chunk_size to the chunk_sizes1 list
    chunk_sizes1.append(chunk_size1)

    # 3p. Append the context_recall_current_chunk_size to the context_recall_scores_chunk_size1 list
    context_recall_scores_chunk_size1.append(context_recall_current_chunk_size)

# 4. After the loop, print the chunk_sizes1 and context_recall_scores_chunk_size1 lists
print("Chunk Sizes:", chunk_sizes1)
print("Context Recall Scores (Chunk Size Variation):",[f"{score:.2f}" for score in context_recall_scores_chunk_size1])

In [ ]:
chunk_sizes = []
context_recall_scores_chunk_size = []

for chunk_size in range(50, 501, 50):
    # 3a. Create RecursiveCharacterTextSplitter documents
    current_chunk_size_docs = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=50).create_documents([long_text])

    # 3b. Initialize a new ChromaDB vector store
    current_chunk_size_vectorstore = Chroma.from_documents(documents=current_chunk_size_docs, embedding=embedding_function)

    # 3c. Create a retriever from this newly created vector store
    current_retriever = current_chunk_size_vectorstore.as_retriever()

    # 3d. Initialize an empty list called retrieved_contexts_current_chunk_size
    retrieved_contexts_current_chunk_size = []

    # 3e. Iterate through each question and answer
    for question, answer in questions_and_answers.items():
        # 3f. Use the current retriever to invoke the question and get the retrieved_documents
        retrieved_documents = current_retriever.invoke(question)

        # 3g. Append a dictionary containing the question, answer, and retrieved_documents
        retrieved_contexts_current_chunk_size.append({
            'question': question,
            'answer': answer,
            'contexts': retrieved_documents
        })

    # 3h. Initialize an empty list called answer_presence_current_chunk_size
    answer_presence_current_chunk_size = []

    # 3i. Iterate through each item in retrieved_contexts_current_chunk_size
    for item in retrieved_contexts_current_chunk_size:
        # 3j. Convert the answer and the page_content of all contexts to lowercase
        answer_lower = item['answer'].lower()
        contexts_text_lower = " ".join([doc.page_content.lower() for doc in item['contexts']])

        # 3k. Check if the lowercase answer is present in the concatenated lowercase contexts_text
        # 3l. Append True or False to answer_presence_current_chunk_size based on the check
        if answer_lower in contexts_text_lower:
            answer_presence_current_chunk_size.append(True)
        else:
            answer_presence_current_chunk_size.append(False)

    # 3m. Calculate the num_answers_found_current_chunk_size
    num_answers_found_current_chunk_size = sum(answer_presence_current_chunk_size)

    # 3n. Calculate the context_recall_current_chunk_size
    context_recall_current_chunk_size = num_answers_found_current_chunk_size / len(questions_and_answers)

    # 3o. Append the current chunk_size to the chunk_sizes list
    chunk_sizes.append(chunk_size)

    # 3p. Append the context_recall_current_chunk_size to the context_recall_scores_chunk_size list
    context_recall_scores_chunk_size.append(context_recall_current_chunk_size)

# 4. After the loop, print the chunk_sizes and context_recall_scores_chunk_size lists
print("Chunk Sizes:", chunk_sizes)
print("Context Recall Scores (Chunk Size Variation):",[f"{score:.2f}" for score in context_recall_scores_chunk_size])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))
plt.plot(chunk_sizes, context_recall_scores_chunk_size, marker='o', linestyle='-', label='overlap=50', color='BLUE')
plt.plot(chunk_sizes1, context_recall_scores_chunk_size1, marker='x', linestyle='--', label='overlap=150', color='red')

plt.title('Context Recall vs. Chunk Size for Different Overlap Strategies')
plt.xlabel('Chunk Size')
plt.ylabel('Context Recall Score')
plt.grid(True)
plt.xticks(chunk_sizes)
plt.ylim(0, 1.05) # Set y-axis limit from 0 to 1 with a small buffer
plt.legend()
plt.show()

## Chunk Sizes (No Overlap)

In [ ]:
vectorstore._client.delete_collection(vectorstore._collection.name)

In [ ]:
chunk_sizes_no_ov = []
context_recall_scores_chunk_size_no_ov = []

for chunk_size in range(1, 252, 25):
    # 3a. Create RecursiveCharacterTextSplitter documents
    current_chunk_size_docs_no_ov = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0).create_documents([long_text])

    # 3b. Initialize a new ChromaDB vector store
    current_chunk_size_vectorstore_no_ov = Chroma.from_documents(documents=current_chunk_size_docs_no_ov, embedding=embedding_function)

    # 3c. Create a retriever from this newly created vector store
    current_retriever_no_ov = current_chunk_size_vectorstore_no_ov.as_retriever()

    # 3d. Initialize an empty list called retrieved_contexts_current_chunk_size_no_ov
    retrieved_contexts_current_chunk_size_no_ov = []

    # 3e. Iterate through each question and answer
    for question, answer in questions_and_answers.items():
        # 3e.i. Use the current retriever to invoke the question and get the retrieved_documents
        retrieved_documents = current_retriever_no_ov.invoke(question)

        # 3e.ii. Append a dictionary containing the question, answer, and retrieved_documents
        retrieved_contexts_current_chunk_size_no_ov.append({
            'question': question,
            'answer': answer,
            'contexts': retrieved_documents
        })

    # 3f. Initialize an empty list called answer_presence_current_chunk_size_no_ov
    answer_presence_current_chunk_size_no_ov = []

    # 3g. Iterate through each item in retrieved_contexts_current_chunk_size_no_ov
    for item in retrieved_contexts_current_chunk_size_no_ov:
        # 3g.i. Convert the answer and the page_content of all contexts to lowercase
        answer_lower = item['answer'].lower()
        contexts_text_lower = " ".join([doc.page_content.lower() for doc in item['contexts']])

        # 3g.ii. Check if the lowercase answer is present in the concatenated lowercase contexts_text
        # 3g.iii. Append True or False to answer_presence_current_chunk_size_no_ov based on the check
        if answer_lower in contexts_text_lower:
            answer_presence_current_chunk_size_no_ov.append(True)
        else:
            answer_presence_current_chunk_size_no_ov.append(False)

    # 3h. Calculate the num_answers_found_current_chunk_size_no_ov
    num_answers_found_current_chunk_size_no_ov = sum(answer_presence_current_chunk_size_no_ov)

    # 3i. Calculate the context_recall_current_chunk_size_no_ov
    context_recall_current_chunk_size_no_ov = num_answers_found_current_chunk_size_no_ov / len(questions_and_answers)

    # 3j. Append the current chunk_size to the chunk_sizes_no_ov list
    chunk_sizes_no_ov.append(chunk_size)

    # 3k. Append the context_recall_current_chunk_size_no_ov to the context_recall_scores_chunk_size_no_ov list
    context_recall_scores_chunk_size_no_ov.append(context_recall_current_chunk_size_no_ov)

# 4. After the loop, print the chunk_sizes_no_ov and context_recall_scores_chunk_size_no_ov lists
print("Chunk Sizes (No Overlap):", chunk_sizes_no_ov)
print("Context Recall Scores (Chunk Size Variation, No Overlap):",[f"{score:.2f}" for score in context_recall_scores_chunk_size_no_ov])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(chunk_sizes_no_ov, context_recall_scores_chunk_size_no_ov, marker='o', linestyle='-')
plt.title('Context Recall vs. Chunk Size (Chunk Overlap = 0)')
plt.xlabel('Chunk Size')
plt.ylabel('Context Recall Score')
plt.grid(True)
plt.xticks(chunk_sizes_no_ov)
plt.ylim(0, 1.05) # Set y-axis limit from 0 to 1 with a small buffer
plt.show()

## Chunk Sizes no Overlap and Chunk Sizes Overlap

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))
plt.plot(chunk_sizes, context_recall_scores_chunk_size, marker='o', linestyle='-', label='With Overlap (overlap=50)')
plt.plot(chunk_sizes, context_recall_scores_chunk_size, marker='o', linestyle='-', label='With Overlap (overlap=150)')
plt.plot(chunk_sizes_no_ov, context_recall_scores_chunk_size_no_ov, marker='x', linestyle='--', label='Without Overlap (overlap=0)', color='red')

plt.title('Context Recall vs. Chunk Size for Different Overlap Strategies')
plt.xlabel('Chunk Size')
plt.ylabel('Context Recall Score')
plt.grid(True)
plt.xticks(chunk_sizes)
plt.ylim(0, 1.05) # Set y-axis limit from 0 to 1 with a small buffer
plt.legend()
plt.show()

## Task 4
### 1. Інформаційна щільність та чистота контексту
*   Семантичне розбиття базується на змісті (абзацах, реченнях або логічних переходах), тоді як фіксоване — на жорсткому ліміті символів. Фіксоване розбиття часто розрізає думку навпіл, створюючи шум: уривки речень на межах блоків не несуть сенсу самі по собі. Це розмиває векторне представлення фрагмента. Натомість семантичне розбиття гарантує, що кожен блок є цілісною ідеєю. Це підвищує точність пошуку, оскільки вектор такого блоку чітко відповідає певній темі, а не є сумішшю двох різних ідей.
### 2. Проблема втраченої середини та перекриття (Overlap)
*   Проблема Lost in the Middle полягає в тому, що моделі найкраще сприймають інформацію на початку або в кінці контексту, часто ігноруючи дані в центрі. Збільшення перекриття (overlap) безпосередньо допомагає зберегти контекст сутностей на межах блоків. Якщо ім’я суб’єкта вказано в кінці одного блоку, а його дія — на початку іншого, без перекриття система може знайти лише дію, не знаючи, хто її вчинив. Перекриття дозволяє зберегти зв'язок між суб'єктом і дією в обох блоках, що покращує якість пошуку, хоча й не вирішує проблему сприйняття моделі, якщо ви подаєте їй занадто багато блоків одночасно.
### 3. Економічна ефективність стратегій
*   Вибір стратегії прямо впливає на вартість запиту. Фіксована стратегія з малим перекриттям має найнижчу ціну через мінімальну надмірність тексту. Фіксована стратегія з великим перекриттям коштує дорожче, бо ви платите за одні й ті самі речення, що дублюються в різних блоках. Семантична стратегія зазвичай є найефективнішою: хоча окремі блоки можуть бути довшими, вони набагато релевантніші. Це дозволяє передавати в модель меншу кількість фрагментів (менше значення K) для отримання якісної відповіді, що зрештою економить кошти на кожному запиті.
